In [ ]:
suppressPackageStartupMessages({
  library(tidyverse)
  library(lubridate)
  library(readxl)
})

proj_dir <- "/Users/siddarthchilukuri/Desktop/443/project"
data_dir <- file.path(proj_dir, "data")
docs_dir <- file.path(proj_dir, "docs")
if (!dir.exists(docs_dir)) dir.create(docs_dir, recursive = TRUE)

csv_files <- list.files(data_dir, pattern = "\\.csv$", full.names = TRUE)
xlsx_files <- list.files(data_dir, pattern = "\\.xlsx$", full.names = TRUE)

cat("CSV files found:", length(csv_files), "\n")
cat("XLSX files found:", length(xlsx_files), "\n")

CSV files found: 9 
XLSX files found: 2 


In [ ]:
data_dictionary <- tibble(
  series_id = c(
    "cpi_can", "cpi_us", "oil_wti", "u6_us", "ppi_us",
    "mxp_us", "exrate_can", "gdp_can", "gdp_us",
    "ippi_can", "unemp_can"
  ),
  file_name = c(
    "cpi_canada_t.csv", "cpi_usa.csv", "crude_oil.csv", "labour_usa.csv", "ppi_usa (2).csv",
    "mxp_usa.csv", "can_exchange_rate.csv", "real_gdp_canada_t.csv", "real_gdp_usa.csv",
    "ippi_can.xlsx", "labour_canada_t.xlsx"
  ),
  country = c(
    "Canada", "United States", "Global", "United States", "United States",
    "United States", "Canada", "Canada", "United States",
    "Canada", "Canada"
  ),
  variable_description = c(
    "Consumer Price Index, all-items, seasonally adjusted",
    "Consumer Price Index, all-items, seasonally adjusted",
    "WTI crude oil spot price",
    "U-6 unemployment rate",
    "Producer price index proxy",
    "US import/export price index (MXP)",
    "CAD effective exchange rate (broad basket)",
    "Monthly real GDP series",
    "Brave-Butters-Kelley real GDP series",
    "Industrial Product Price Index",
    "Unemployment rate (Canada)"
  ),
  frequency = "Monthly",
  unit = NA_character_,
  transformation = NA_character_,
  lag_rule = "Use lagged predictors only (starting with lag 1)",
  first_date = NA_character_,
  last_date = NA_character_,
  notes = NA_character_
 )

read_date_range <- function(path) {
  ext <- tools::file_ext(path)
  if (ext == "csv") {
    df <- suppressWarnings(readr::read_csv(path, show_col_types = FALSE))
  } else if (ext == "xlsx") {
    df <- suppressWarnings(readxl::read_excel(path))
  } else {
    return(c(NA_character_, NA_character_))
  }

  date_candidates <- names(df)[stringr::str_detect(names(df), stringr::regex("date|time|period|ref", ignore_case = TRUE))]
  if (length(date_candidates) == 0) return(c(NA_character_, NA_character_))

  raw_date <- as.character(df[[date_candidates[1]]])
  parsed <- suppressWarnings(lubridate::parse_date_time(
    raw_date,
    orders = c("ymd", "Ymd", "b-y", "Y b", "Y b d", "b Y", "Y-m-d")
  ))

  if (all(is.na(parsed))) return(c(NA_character_, NA_character_))
  c(as.character(min(parsed, na.rm = TRUE)), as.character(max(parsed, na.rm = TRUE)))
}

for (i in seq_len(nrow(data_dictionary))) {
  f <- file.path(data_dir, data_dictionary$file_name[i])
  if (file.exists(f)) {
    rng <- read_date_range(f)
    data_dictionary$first_date[i] <- rng[1]
    data_dictionary$last_date[i] <- rng[2]
  }
}

out_path <- file.path(docs_dir, "data_dictionary_template.csv")
readr::write_csv(data_dictionary, out_path)
cat("Wrote:", out_path, "\n")
print(data_dictionary)

Wrote: /Users/siddarthchilukuri/Desktop/443/project/docs/data_dictionary_template.csv 
# A tibble: 11 × 11
   series_id  file_name             country variable_description frequency unit 
   <chr>      <chr>                 <chr>   <chr>                <chr>     <chr>
 1 cpi_can    cpi_canada_t.csv      Canada  Consumer Price Inde… Monthly   NA   
 2 cpi_us     cpi_usa.csv           United… Consumer Price Inde… Monthly   NA   
 3 oil_wti    crude_oil.csv         Global  WTI crude oil spot … Monthly   NA   
 4 u6_us      labour_usa.csv        United… U-6 unemployment ra… Monthly   NA   
 5 ppi_us     ppi_usa (2).csv       United… Producer price inde… Monthly   NA   
 6 mxp_us     mxp_usa.csv           United… US import/export pr… Monthly   NA   
 7 exrate_can can_exchange_rate.csv Canada  CAD effective excha… Monthly   NA   
 8 gdp_can    real_gdp_canada_t.csv Canada  Monthly real GDP se… Monthly   NA   
 9 gdp_us     real_gdp_usa.csv      United… Brave-Butters-Kelle… Monthly   NA   
10

In [ ]:
dict_path <- file.path(docs_dir, "data_dictionary_template.csv")
dd <- readr::read_csv(dict_path, show_col_types = FALSE)

dd_filled <- dd %>%
  mutate(
    unit = case_when(
      series_id %in% c("cpi_can", "cpi_us", "ppi_us", "mxp_us", "ippi_can") ~ "Index level",
      series_id == "oil_wti" ~ "USD per barrel",
      series_id %in% c("u6_us", "unemp_can", "gdp_us") ~ "Percent/rate",
      series_id == "exrate_can" ~ "Index (broad basket)",
      series_id == "gdp_can" ~ "Millions of chained dollars",
      TRUE ~ unit
    ),
    transformation = case_when(
      series_id %in% c("cpi_can", "cpi_us") ~ "Target: inflation_t = log(CPI_t) - log(CPI_t-1)",
      series_id %in% c("oil_wti", "ppi_us", "mxp_us", "exrate_can", "gdp_can", "ippi_can") ~ "Predictor: monthly log-difference",
      series_id %in% c("u6_us", "unemp_can") ~ "Predictor: lagged level (robustness: first difference)",
      series_id == "gdp_us" ~ "Use as provided (rate series), then lag",
      TRUE ~ transformation
    ),
    notes = case_when(
      series_id %in% c("cpi_can", "cpi_us") ~ "Main inflation target series.",
      series_id == "gdp_can" ~ "Monthly level series; confirm latest units when updating data.",
      series_id == "gdp_us" ~ "Already looks like a growth/rate series.",
      series_id %in% c("u6_us", "unemp_can") ~ "Labour market slack variable.",
      series_id %in% c("ppi_us", "ippi_can") ~ "Cost pressure proxy.",
      series_id %in% c("mxp_us", "exrate_can") ~ "Imported inflation channel.",
      series_id == "oil_wti" ~ "Global energy shock control.",
      TRUE ~ notes
    )
  )

readr::write_csv(dd_filled, dict_path)
cat("Updated:", dict_path, "\n")
print(dd_filled %>% select(series_id, unit, transformation, notes))

Updated: /Users/siddarthchilukuri/Desktop/443/project/docs/data_dictionary_template.csv 
# A tibble: 11 × 4
   series_id  unit                        transformation                   notes
   <chr>      <chr>                       <chr>                            <chr>
 1 cpi_can    Index level                 Target: inflation_t = log(CPI_t… Main…
 2 cpi_us     Index level                 Target: inflation_t = log(CPI_t… Main…
 3 oil_wti    USD per barrel              Predictor: monthly log-differen… Glob…
 4 u6_us      Percent/rate                Predictor: lagged level (robust… Labo…
 5 ppi_us     Index level                 Predictor: monthly log-differen… Cost…
 6 mxp_us     Index level                 Predictor: monthly log-differen… Impo…
 7 exrate_can Index (broad basket)        Predictor: monthly log-differen… Impo…
 8 gdp_can    Millions of chained dollars Predictor: monthly log-differen… Mont…
 9 gdp_us     Percent/rate                Use as provided (rate series), … Alre…
1

In [ ]:
dict_path <- file.path(docs_dir, "data_dictionary_template.csv")
dd_final <- readr::read_csv(dict_path, show_col_types = FALSE)

dd_final <- dd_final %>%
  mutate(
    source_link = case_when(
      series_id == "oil_wti" ~ "https://fred.stlouisfed.org/series/MCOILWTICO",
      series_id == "cpi_us" ~ "https://fred.stlouisfed.org/series/CPIAUCSL",
      series_id == "u6_us" ~ "https://fred.stlouisfed.org/series/U6RATE",
      series_id == "gdp_us" ~ "https://fred.stlouisfed.org/series/BBKMGDP",
      series_id == "mxp_us" ~ "https://www.bls.gov/mxp/",
      series_id == "cpi_can" ~ "https://www150.statcan.gc.ca/t1/tbl1/en/tv.action?pid=1810000601",
      series_id == "exrate_can" ~ "https://www150.statcan.gc.ca/t1/tbl1/en/tv.action?pid=3310016301",
      series_id == "ippi_can" ~ "https://www150.statcan.gc.ca/t1/tbl1/en/tv.action?pid=1810026501",
      series_id == "unemp_can" ~ "https://www150.statcan.gc.ca/t1/tbl1/en/tv.action?pid=1410028701",
      series_id == "gdp_can" ~ "https://www150.statcan.gc.ca/t1/tbl1/en/tv.action?pid=3610043401",
      series_id == "ppi_us" ~ "https://fred.stlouisfed.org/series/WPSFD49208",
      TRUE ~ NA_character_
    ),
    unit = case_when(
      series_id == "oil_wti" ~ "Dollars per barrel (FRED: NSA)",
      series_id == "cpi_us" ~ "Index 1982-1984 = 100 (FRED: SA)",
      series_id == "u6_us" ~ "Percent (FRED: SA)",
      series_id == "gdp_us" ~ "Annualized percent change from preceding period (FRED: SA)",
      series_id == "mxp_us" ~ "Import/Export price index level (BLS MXP)",
      series_id == "cpi_can" ~ "CPI index level (StatCan SA monthly table)",
      series_id == "exrate_can" ~ "Nominal effective exchange rate index, broad basket",
      series_id == "ippi_can" ~ "Industrial product price index level",
      series_id == "unemp_can" ~ "Unemployment rate, percent",
      series_id == "gdp_can" ~ "x 1,000,000 chained (2017) dollars, SAAR",
      series_id == "ppi_us" ~ "Index 1982 = 100 (FRED: SA)",
      TRUE ~ unit
    ),
    transformation = case_when(
      series_id %in% c("cpi_can", "cpi_us") ~ "Target: inflation_t = log(CPI_t) - log(CPI_t-1)",
      series_id %in% c("oil_wti", "mxp_us", "exrate_can", "ippi_can", "ppi_us", "gdp_can") ~ "Predictor: log-difference, then lag 1+",
      series_id == "gdp_us" ~ "Predictor: keep as rate series, then lag 1+",
      series_id %in% c("u6_us", "unemp_can") ~ "Predictor: lagged level; robustness check with first difference",
      TRUE ~ transformation
    ),
    notes = case_when(
      series_id == "cpi_us" ~ "FRED monthly SA CPI series.",
      series_id == "oil_wti" ~ "FRED monthly NSA WTI series.",
      series_id == "u6_us" ~ "FRED monthly SA U-6 series.",
      series_id == "gdp_us" ~ "FRED monthly SA annualized growth series.",
      series_id == "mxp_us" ~ "BLS MXP program source; keep selected series details in appendix.",
      series_id == "cpi_can" ~ "StatCan table 18-10-0006-01.",
      series_id == "exrate_can" ~ "StatCan table 33-10-0163-01.",
      series_id == "ippi_can" ~ "StatCan table 18-10-0265-01.",
      series_id == "unemp_can" ~ "StatCan table 14-10-0287-01.",
      series_id == "gdp_can" ~ "StatCan table 36-10-0434-01.",
      series_id == "ppi_us" ~ "FRED WPSFD49208: SA monthly PPI (finished goods less energy).",
      TRUE ~ notes
    )
  )

out_final <- file.path(docs_dir, "data_dictionary_final.csv")
readr::write_csv(dd_final, out_final)
readr::write_csv(dd_final, dict_path)
cat("Updated dictionary files:\n", dict_path, "\n", out_final, "\n")
print(dd_final %>% select(series_id, unit, transformation, notes, source_link))

Updated dictionary files:
 /Users/siddarthchilukuri/Desktop/443/project/docs/data_dictionary_template.csv 
 /Users/siddarthchilukuri/Desktop/443/project/docs/data_dictionary_final.csv 
# A tibble: 11 × 5
   series_id  unit                              transformation notes source_link
   <chr>      <chr>                             <chr>          <chr> <chr>      
 1 cpi_can    CPI index level (StatCan SA mont… Target: infla… Stat… https://ww…
 2 cpi_us     Index 1982-1984 = 100 (FRED: SA)  Target: infla… FRED… https://fr…
 3 oil_wti    Dollars per barrel (FRED: NSA)    Predictor: lo… FRED… https://fr…
 4 u6_us      Percent (FRED: SA)                Predictor: la… FRED… https://fr…
 5 ppi_us     Index 1982 = 100 (FRED: SA)       Predictor: lo… FRED… https://fr…
 6 mxp_us     Import/Export price index level … Predictor: lo… BLS … https://ww…
 7 exrate_can Nominal effective exchange rate … Predictor: lo… Stat… https://ww…
 8 gdp_can    x 1,000,000 chained (2017) dolla… Predictor: lo… Stat

In [ ]:
if (!dir.exists(file.path(proj_dir, "data_clean"))) dir.create(file.path(proj_dir, "data_clean"), recursive = TRUE)

parse_month_date <- function(x) {
  x <- as.character(x)
  out <- suppressWarnings(lubridate::parse_date_time(
    x,
    orders = c("ymd", "Ymd", "b-y", "b Y", "Y b", "Y-m-d", "Y/m/d")
  ))
  as.Date(out)
}

read_series <- function(series_id, file_name, date_col, value_col, file_type = "csv") {
  path <- file.path(data_dir, file_name)
  if (file_type == "xlsx") {
    df <- readxl::read_excel(path)
  } else {
    df <- readr::read_csv(path, show_col_types = FALSE)
  }

  tibble(
    series_id = series_id,
    date_raw = df[[date_col]],
    value = suppressWarnings(as.numeric(df[[value_col]]))
  ) %>%
    mutate(date = parse_month_date(date_raw)) %>%
    select(series_id, date, value, date_raw) %>%
    arrange(date)
}

series_data <- list(
  cpi_can = read_series("cpi_can", "cpi_canada_t.csv", "REF_DATE", "VALUE"),
  cpi_us = read_series("cpi_us", "cpi_usa.csv", "observation_date", "CPIAUCSL"),
  oil_wti = read_series("oil_wti", "crude_oil.csv", "observation_date", "MCOILWTICO"),
  u6_us = read_series("u6_us", "labour_usa.csv", "observation_date", "U6RATE"),
  ppi_us = read_series("ppi_us", "ppi_usa (2).csv", "observation_date", "WPSFD49208"),
  mxp_us = read_series("mxp_us", "mxp_usa.csv", "Label", "Value"),
  exrate_can = read_series("exrate_can", "can_exchange_rate.csv", "TIME_PERIOD:Period", "OBS_VALUE:Value"),
  gdp_can = read_series("gdp_can", "real_gdp_canada_t.csv", "REF_DATE", "VALUE"),
  gdp_us = read_series("gdp_us", "real_gdp_usa.csv", "observation_date", "BBKMGDP"),
  ippi_can = read_series("ippi_can", "ippi_can.xlsx", "REF_DATE", "VALUE", file_type = "xlsx"),
  unemp_can = read_series("unemp_can", "labour_canada_t.xlsx", "REF_DATE", "VALUE", file_type = "xlsx")
)

series_data

series_id,date,value,date_raw
<chr>,<date>,<dbl>,<chr>
cpi_can,2004-01-01,103.7,Jan-04
cpi_can,2004-02-01,103.7,Feb-04
cpi_can,2004-03-01,103.7,Mar-04
cpi_can,2004-04-01,104.0,Apr-04
cpi_can,2004-05-01,104.7,May-04
cpi_can,2004-06-01,104.8,Jun-04
cpi_can,2004-07-01,104.8,Jul-04
cpi_can,2004-08-01,104.7,Aug-04
cpi_can,2004-09-01,104.8,Sep-04


In [ ]:
schema_check <- purrr::imap_dfr(series_data, function(df, nm) {
  tibble(
    series_id = nm,
    n_rows = nrow(df),
    n_missing_date = sum(is.na(df$date)),
    n_missing_value = sum(is.na(df$value)),
    n_duplicate_months = sum(duplicated(df$date) & !is.na(df$date)),
    start_date = as.character(min(df$date, na.rm = TRUE)),
    end_date = as.character(max(df$date, na.rm = TRUE)),
    value_min = suppressWarnings(min(df$value, na.rm = TRUE)),
    value_max = suppressWarnings(max(df$value, na.rm = TRUE))
  )
}) %>%
  arrange(series_id)

schema_path <- file.path(docs_dir, "schema_date_check.csv")
readr::write_csv(schema_check, schema_path)

stacked_path <- file.path(proj_dir, "data_clean", "series_stacked.csv")
stacked_df <- bind_rows(purrr::map(series_data, ~ mutate(.x, date_raw = as.character(date_raw))))
readr::write_csv(stacked_df, stacked_path)

cat("Wrote:\n", schema_path, "\n", stacked_path, "\n")
print(schema_check)

Wrote:
 /Users/siddarthchilukuri/Desktop/443/project/docs/schema_date_check.csv 
 /Users/siddarthchilukuri/Desktop/443/project/data_clean/series_stacked.csv 
# A tibble: 11 × 9
   series_id n_rows n_missing_date n_missing_value n_duplicate_months start_date
   <chr>      <int>          <int>           <int>              <int> <chr>     
 1 cpi_can      266              0               0                  0 2004-01-01
 2 cpi_us       266              0               1                  0 2004-01-01
 3 exrate_c…    267              0               0                  0 2003-12-31
 4 gdp_can      265              0               0                  0 2004-01-01
 5 gdp_us       264              0               0                  0 2004-01-01
 6 ippi_can     266              0               0                  0 2004-01-01
 7 mxp_us       265              0               0                  0 2004-01-01
 8 oil_wti      267              0               0                  0 2004-01-01
 9 ppi_us    

In [ ]:
if (!dir.exists(file.path(proj_dir, "data_processed"))) dir.create(file.path(proj_dir, "data_processed"), recursive = TRUE)

panel_wide <- bind_rows(
  purrr::map(series_data, ~ .x %>% mutate(month = as.Date(lubridate::floor_date(date, "month"))) %>% select(series_id, month, value))
 ) %>%
  group_by(series_id, month) %>%
  summarise(value = dplyr::first(value), .groups = "drop") %>%
  tidyr::pivot_wider(names_from = series_id, values_from = value) %>%
  arrange(month)

panel_t <- panel_wide %>%
  mutate(
    infl_can = log(cpi_can) - log(lag(cpi_can)),
    infl_us = log(cpi_us) - log(lag(cpi_us)),
    oil_wti_ld = log(oil_wti) - log(lag(oil_wti)),
    ppi_us_ld = log(ppi_us) - log(lag(ppi_us)),
    mxp_us_ld = log(mxp_us) - log(lag(mxp_us)),
    exrate_can_ld = log(exrate_can) - log(lag(exrate_can)),
    ippi_can_ld = log(ippi_can) - log(lag(ippi_can)),
    gdp_can_ld = log(gdp_can) - log(lag(gdp_can)),
    gdp_us_rate = gdp_us,
    u6_us_lvl = u6_us,
    unemp_can_lvl = unemp_can
  ) %>%
  mutate(
    oil_wti_l1 = lag(oil_wti_ld),
    ppi_us_l1 = lag(ppi_us_ld),
    mxp_us_l1 = lag(mxp_us_ld),
    exrate_can_l1 = lag(exrate_can_ld),
    ippi_can_l1 = lag(ippi_can_ld),
    gdp_can_l1 = lag(gdp_can_ld),
    gdp_us_l1 = lag(gdp_us_rate),
    u6_us_l1 = lag(u6_us_lvl),
    unemp_can_l1 = lag(unemp_can_lvl)
  )

canada_model_raw <- panel_t %>%
  transmute(
    month,
    infl = infl_can,
    gdp_can_l1,
    unemp_can_l1,
    ippi_can_l1,
    oil_wti_l1,
    exrate_can_l1
  )

us_model_raw <- panel_t %>%
  transmute(
    month,
    infl = infl_us,
    gdp_us_l1,
    u6_us_l1,
    ppi_us_l1,
    oil_wti_l1,
    mxp_us_l1
  )

canada_model <- canada_model_raw %>% filter(if_all(-month, ~ !is.na(.)))
us_model <- us_model_raw %>% filter(if_all(-month, ~ !is.na(.)))

common_start <- max(min(canada_model$month), min(us_model$month))
common_end <- min(max(canada_model$month), max(us_model$month))

canada_common <- canada_model %>% filter(month >= common_start, month <= common_end)
us_common <- us_model %>% filter(month >= common_start, month <= common_end)

canada_path <- file.path(proj_dir, "data_processed", "canada_model_full_sample.csv")
us_path <- file.path(proj_dir, "data_processed", "us_model_full_sample.csv")
canada_common_path <- file.path(proj_dir, "data_processed", "canada_model_common_window.csv")
us_common_path <- file.path(proj_dir, "data_processed", "us_model_common_window.csv")

readr::write_csv(canada_model, canada_path)
readr::write_csv(us_model, us_path)
readr::write_csv(canada_common, canada_common_path)
readr::write_csv(us_common, us_common_path)

window_summary <- tibble::tibble(
  metric = c("canada_model_full_sample", "us_model_full_sample", "common_window"),
  start_date = as.character(c(min(canada_model$month), min(us_model$month), common_start)),
  end_date = as.character(c(max(canada_model$month), max(us_model$month), common_end)),
  n_obs_canada = c(nrow(canada_model), NA_integer_, nrow(canada_common)),
  n_obs_us = c(NA_integer_, nrow(us_model), nrow(us_common))
 )

window_path <- file.path(docs_dir, "common_sample_window.csv")
readr::write_csv(window_summary, window_path)

cat("Wrote:\n", canada_path, "\n", us_path, "\n", canada_common_path, "\n", us_common_path, "\n", window_path, "\n")
print(window_summary)

Wrote:
 /Users/siddarthchilukuri/Desktop/443/project/data_processed/canada_model_full_sample.csv 
 /Users/siddarthchilukuri/Desktop/443/project/data_processed/us_model_full_sample.csv 
 /Users/siddarthchilukuri/Desktop/443/project/data_processed/canada_model_common_window.csv 
 /Users/siddarthchilukuri/Desktop/443/project/data_processed/us_model_common_window.csv 
 /Users/siddarthchilukuri/Desktop/443/project/docs/common_sample_window.csv 
# A tibble: 3 × 5
  metric                   start_date end_date   n_obs_canada n_obs_us
  <chr>                    <chr>      <chr>             <int>    <int>
1 canada_model_full_sample 2004-03-01 2026-02-01          264       NA
2 us_model_full_sample     2004-03-01 2026-01-01           NA      260
3 common_window            2004-03-01 2026-01-01          263      260
